# Aula 5 — Redes neuronais: backward propagation

Testes do `backward_propagation` da `DenseLayer` (slide 12) e das camadas de ativação (slide 20).


## DenseLayer.backward_propagation (slides 11–13 ppt da aula6-P-solved.pdf)

Gradient check: compara o dE/dW calculado pelo `backward_propagation` com uma derivada numérica. Se a fórmula estiver certa, a diferença é praticamente zero.



In [1]:
import numpy as np
from si.neural_networks.layers import DenseLayer

# optimizer "falso": não mexe nos pesos, só guarda o último gradiente que recebeu
# (o SGD e o Adam só chegam na próxima aula)
class GuardaGradiente:
    def update(self, w, grad):
        self.grad = grad
        return w

# MSE em numpy (as funções de loss também só chegam na próxima aula)
mse = lambda Y, P: np.mean((Y - P) ** 2)
# dE/dY: divide por Y.size porque a MSE é uma média de todos os elementos
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size



In [2]:
np.random.seed(0)
X, Y = np.random.rand(4, 3), np.random.rand(4, 2)  # 4 amostras, 3 features, 2 outputs

layer = DenseLayer(2, (3,)).initialize(GuardaGradiente())
out = layer.forward_propagation(X, training=True)
layer.backward_propagation(mse_derivative(Y, out))
analytic = layer.w_opt.grad  # dE/dW calculado pelo backward_propagation
# (lê-se de layer.w_opt e não do objeto original: o deepcopy do initialize criou uma cópia)

# derivada numérica: mexe cada peso um bocadinho para cima e para baixo e vê quanto a loss muda
numeric, eps = np.zeros_like(layer.weights), 1e-6
for i in range(3):
    for j in range(2):
        layer.weights[i, j] += eps
        lp = mse(Y, layer.forward_propagation(X, True))  # loss com w + eps
        layer.weights[i, j] -= 2 * eps
        lm = mse(Y, layer.forward_propagation(X, True))  # loss com w - eps
        layer.weights[i, j] += eps                         # volta ao valor original
        numeric[i, j] = (lp - lm) / (2 * eps)              # declive entre os dois pontos

print("dE/dW (backward):\n", analytic.round(6))
print("dE/dW (numérico):\n", numeric.round(6))
print("diferença máxima:", np.abs(numeric - analytic).max())  # ~1e-11

dE/dW (backward):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
dE/dW (numérico):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
diferença máxima: 1.513400516017782e-11


**Conclusão (DenseLayer.backward_propagation):** o dE/dW calculado pelo `backward_propagation` é igual à derivada numérica (diferença da ordem de 1e-11), por isso a fórmula dE/dW = Xᵀ·dE/dY está bem implementada.

- O `backward_propagation` calcula 3 gradientes, cada um com a shape daquilo a que se refere: dE/dX = dE/dY·Wᵀ (shape de X, passa para a camada anterior), dE/dW = Xᵀ·dE/dY (shape de W) e dE/dB = soma de dE/dY nas amostras (shape de B).
- O dE/dX tem de ser calculado antes de atualizar os pesos, senão usava os pesos novos.
- O teste usa um optimizer "falso" que não mexe nos pesos, porque o SGD, o Adam e as funções de loss só chegam na próxima aula.

## Camadas de ativação (slides 14–20)

Implementadas em `src/si/neural_networks/activation.py`: classe base abstrata `ActivationLayer`, `ReLUActivation` e `SigmoidActivation`. O teste compara a derivada de cada ativação com a derivada numérica.

In [3]:
from si.neural_networks.activation import ReLUActivation, SigmoidActivation

# valores de teste: evitam o 0, onde a derivada da ReLU não existe
x, eps = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]), 1e-6

for act in [SigmoidActivation(), ReLUActivation()]:
    # derivada numérica: declive entre f(x - eps) e f(x + eps)
    num = (act.activation_function(x + eps) - act.activation_function(x - eps)) / (2 * eps)
    print(act.layer_name())
    print("  f(x):  ", act.activation_function(x).round(4))
    print("  f'(x): ", act.derivative(x).round(4))
    print("  diferença máxima:", np.abs(num - act.derivative(x)).max())  # ~1e-10

SigmoidActivation
  f(x):   [0.0759 0.2315 0.4256 0.5987 0.7503 0.937 ]
  f'(x):  [0.0701 0.1779 0.2445 0.2403 0.1874 0.059 ]
  diferença máxima: 6.56039597646263e-11
ReLUActivation
  f(x):   [0.  0.  0.  0.4 1.1 2.7]
  f'(x):  [0 0 0 1 1 1]
  diferença máxima: 1.397779669787269e-10


**Conclusão (ativações):** as derivadas da `SigmoidActivation` e da `ReLUActivation` coincidem com as derivadas numéricas (diferença da ordem de 1e-10).

- A ReLU põe os negativos a zero e deixa os positivos iguais; a derivada é 0 ou 1, por isso não encolhe o erro nos neurónios ativos.
- A sigmoide comprime os valores para ]0, 1[; a derivada f(x)·(1 − f(x)) é no máximo 0.25, por isso, com muitas camadas, o erro vai ficando cada vez mais pequeno (vanishing gradient).
- No backward, a multiplicação é elemento a elemento (`*`), porque cada output de uma ativação depende só do seu próprio input.


## Exercício 12 — Dropout (avaliação)

Camada `Dropout` em `src/si/neural_networks/layers.py`. Teste com um input aleatório: em treino, cerca de `probability` dos valores devem ficar a 0 e os restantes devem ser multiplicados por 1 / (1 − probability); em previsão, o output deve ser igual ao input.

In [4]:
from si.neural_networks.layers import Dropout

np.random.seed(1)
X = np.random.rand(3, 4)  # 3 amostras, 4 features
dropout = Dropout(probability=0.5)

# treino: aplica a máscara e reescala os valores mantidos
out_treino = dropout.forward_propagation(X, training=True)
print("input:\n", X.round(3))
print("máscara:\n", dropout.mask)
print("output em treino:\n", out_treino.round(3))

# os mantidos foram multiplicados por 1 / (1 - 0.5) = 2 e os desligados ficaram a 0
mantidos = dropout.mask == 1
print("mantidos = input * 2:", np.allclose(out_treino[mantidos], X[mantidos] * 2))
print("desligados = 0:", np.all(out_treino[~mantidos] == 0))

# backward: o erro só passa pelos neurónios mantidos
erro = np.ones_like(X)
print("backward = máscara:", np.array_equal(dropout.backward_propagation(erro), dropout.mask))

# previsão: o input passa igual
out_previsao = dropout.forward_propagation(X, training=False)
print("previsão igual ao input:", np.array_equal(out_previsao, X))

dropout.set_input_shape((4,))
print(dropout.layer_name(), "| output_shape:", dropout.output_shape(), "| parâmetros:", dropout.parameters())

input:
 [[0.417 0.72  0.    0.302]
 [0.147 0.092 0.186 0.346]
 [0.397 0.539 0.419 0.685]]
máscara:
 [[0 1 0 1]
 [0 1 0 0]
 [1 1 0 1]]
output em treino:
 [[0.    1.441 0.    0.605]
 [0.    0.185 0.    0.   ]
 [0.794 1.078 0.    1.37 ]]
mantidos = input * 2: True
desligados = 0: True
backward = máscara: True
previsão igual ao input: True
Dropout | output_shape: (4,) | parâmetros: 0


In [5]:
# com muitos valores, a fração desligada aproxima-se de probability
grande = Dropout(probability=0.3)
out = grande.forward_propagation(np.ones((1000, 100)), training=True)
print("fração desligada:", round(1 - grande.mask.mean(), 3))  # ~0.3
print("média do output:", round(out.mean(), 3))              # ~1.0: o fator de escala compensa os zeros

fração desligada: 0.3
média do output: 1.001


**Conclusão (Exercício 12):** a camada `Dropout` comporta-se como pretendido.

- Em treino, cada valor é desligado (posto a 0) com probabilidade `probability`: com 100 000 valores e `probability = 0.3`, a fração desligada foi ≈ 0.3.
- Os valores mantidos são multiplicados por 1 / (1 − probability) (×2 com p = 0.5), por isso a média do output fica ≈ igual à do input (≈ 1.0): a camada seguinte recebe valores da mesma ordem em treino e em previsão.
- Em previsão, o input passa sem alterações (usa-se a rede toda).
- No backward, o erro é multiplicado pela máscara: os neurónios desligados não contribuíram para o output, por isso não recebem erro.
- Não muda a shape e não tem parâmetros treináveis (`output_shape` = input_shape, `parameters` = 0).

## Exercício 13 — TanhActivation e SoftmaxActivation (avaliação)

Classes em `src/si/neural_networks/activation.py`. O teste compara as derivadas com as derivadas numéricas e confirma que a softmax devolve probabilidades (cada amostra soma 1) e é estável com valores grandes.


In [6]:
from si.neural_networks.activation import TanhActivation, SoftmaxActivation

# 1 amostra com 6 valores (2D: linhas = amostras, colunas = neurónios)
x, eps = np.array([[-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]]), 1e-6


def derivada_numerica(act, x):
    # mexe um valor de cada vez e mede quanto muda o SEU output -> f'(x_i) em cada posição
    # (no softmax não dá para mexer em todos ao mesmo tempo, como no teste do slide 20:
    # somar a mesma constante a todos os valores não muda o resultado da softmax)
    num = np.zeros_like(x)
    for i in range(x.shape[1]):
        x_mais, x_menos = x.copy(), x.copy()
        x_mais[0, i] += eps
        x_menos[0, i] -= eps
        num[0, i] = (act.activation_function(x_mais)[0, i] - act.activation_function(x_menos)[0, i]) / (2 * eps)
    return num


for act in [TanhActivation(), SoftmaxActivation()]:
    print(act.layer_name())
    print("  f(x):  ", act.activation_function(x).round(4))
    print("  f'(x): ", act.derivative(x).round(4))
    print("  diferença máxima:", np.abs(derivada_numerica(act, x) - act.derivative(x)).max())

TanhActivation
  f(x):   [[-0.9866 -0.8337 -0.2913  0.3799  0.8005  0.991 ]]
  f'(x):  [[0.0266 0.305  0.9151 0.8556 0.3592 0.0179]]
  diferença máxima: 3.338995746560158e-11
SoftmaxActivation
  f(x):   [[0.004  0.0147 0.0361 0.0728 0.1465 0.7258]]
  f'(x):  [[0.004  0.0145 0.0348 0.0675 0.1251 0.199 ]]
  diferença máxima: 2.901348605810483e-11


In [7]:
softmax = SoftmaxActivation()

# cada linha (amostra) soma 1: é uma distribuição de probabilidades pelas classes
scores = np.array([[2.0, 1.0, 0.1],
                   [1.0, 3.0, 0.5]])
probs = softmax.activation_function(scores)
print("probabilidades:\n", probs.round(4))
print("soma de cada linha:", probs.sum(axis=1))

# estabilidade: np.exp(1000) dava inf, mas a versão estável funciona
# e dá o mesmo que [0, 1, 2], porque só interessam as diferenças entre os valores
grandes = np.array([[1000.0, 1001.0, 1002.0]])
print("valores grandes:", softmax.activation_function(grandes).round(4))
print("igual a [0, 1, 2]:", np.allclose(softmax.activation_function(grandes),
                                        softmax.activation_function(np.array([[0.0, 1.0, 2.0]]))))

probabilidades:
 [[0.659  0.2424 0.0986]
 [0.1112 0.8214 0.0674]]
soma de cada linha: [1. 1.]
valores grandes: [[0.09   0.2447 0.6652]]
igual a [0, 1, 2]: True


**Conclusão (Exercício 13):** as derivadas da `TanhActivation` e da `SoftmaxActivation` coincidem com as derivadas numéricas (diferença da ordem de 1e-11).

- A tanh comprime os valores para ]−1, 1[ e é centrada em zero; a derivada 1 − f(x)² chega a 1 (em x = 0), contra 0.25 na sigmoide.
- A softmax transforma os scores de cada amostra em probabilidades: cada linha soma 1.
- A versão estável (subtrair o máximo de cada amostra) dá o mesmo resultado, mas evita overflow: com [1000, 1001, 1002] funciona e dá o mesmo que [0, 1, 2].
- A derivada pedida para a softmax, f(x)·(1 − f(x)), é a derivada de cada output em relação ao seu próprio input. Por isso o teste mexe num valor de cada vez: se mexesse em todos ao mesmo tempo, a softmax não mudava.
